In [57]:
import json

from protest_impact.util import project_root

with open(
    project_root / "data" / "news" / "scrapable_mediacloud_newspapers_v1.json"
) as f:
    newspapers = json.load(f)

In [58]:
newspapers[:5]

[{'name': 'hna', 'media_id': 39267},
 {'name': 'tz.de', 'media_id': 143671},
 {'name': 'merkur-online', 'media_id': 39005},
 {'name': 'Christ & Welt: ZEIT für Glaube, Geist und Gesellschaft',
  'media_id': 385524},
 {'name': 'wn.de', 'media_id': 145368}]

In [59]:
def querify(s):
    if len(s.split()) == 1:
        return s
    else:
        return f'"{s}"'

In [66]:
from datetime import datetime
from os import environ
from shutil import rmtree

import pandas as pd
from dotenv import load_dotenv

from protest_impact.util.cache import get

load_dotenv()


def get_counts(newspaper, keyword):
    query = querify(keyword)
    media_id = newspaper["media_id"]
    if not get.check_call_in_cache(
        "https://api.mediacloud.org/api/v2/stories_public/count/",
        params={
            "q": query,
            "fq": f"media_id:{media_id}",
            "split": True,
            "split_period": "day",
            "key": environ["MEDIACLOUD_API_KEY"],
        },
        headers={"Accept": "application/json"},
    ):
        return None
    response = get(
        "https://api.mediacloud.org/api/v2/stories_public/count/",
        params={
            "q": query,
            "fq": f"media_id:{media_id}",
            "split": True,
            "split_period": "day",
            "key": environ["MEDIACLOUD_API_KEY"],
        },
        headers={"Accept": "application/json"},
    )
    result = response.json()
    if "counts" not in result:
        print("⚠️ missing", keyword)
        return None
    df = pd.DataFrame(result["counts"])
    if len(df) == 0:
        return None
    df["date"] = pd.to_datetime(df["date"])
    df = df[(df["date"].dt.year >= 2010) & (df["date"] < datetime.now())]
    df["media_id"] = media_id
    df["name"] = newspaper["name"]
    df["keyword"] = keyword
    return df

In [67]:
import yaml

with open(project_root / "protest_impact" / "data" / "protests" / "config.yaml") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

In [68]:
movement_kws = config["movement_keywords"]
test_kw = movement_kws["climate"]["movement"][0]
test_kw

'fridays for future'

In [69]:
querify("fridays for future")

'"fridays for future"'

In [70]:
get_counts(newspapers[0], test_kw).head()

,count,date,media_id,name,keyword
0,1,2018-12-14,39267,hna,fridays for future
1,1,2019-01-31,39267,hna,fridays for future
2,1,2019-02-02,39267,hna,fridays for future
3,1,2019-03-03,39267,hna,fridays for future
4,1,2019-03-14,39267,hna,fridays for future


In [71]:
from tqdm.notebook import tqdm

dfs = []
for movement in list(movement_kws):
    print(movement)
    if "topic" not in movement_kws[movement]:
        continue
    for kw in tqdm(list(movement_kws[movement]["topic"])):
        for newspaper in newspapers:
            counts = get_counts(newspaper, kw)
            if counts is not None:
                dfs.append(counts)
df = pd.concat(dfs)
df.to_csv("topic_counts.csv", index=False)
df.head()

climate


  0%|          | 0/10 [00:00<?, ?it/s]

racism


  0%|          | 0/10 [00:00<?, ?it/s]

⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
⚠️ missing nsu
feminism


...

  0%|          | 0/4 [00:00<?, ?it/s]

international


  0%|          | 0/37 [00:00<?, ?it/s]

⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing hong kong
⚠️ missing ho

,count,date,media_id,name,keyword
0,1,2013-03-13,39267,hna,klima
1,1,2013-03-24,39267,hna,klima
2,1,2013-03-29,39267,hna,klima
3,1,2013-04-04,39267,hna,klima
4,1,2013-04-15,39267,hna,klima
